Решим задачу регрессии для IC50

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_excel(
    "Данные_для_курсовои_Классическое_МО.xlsx"
)
df = df.drop(columns=["Unnamed: 0"])
df = df.fillna(df.median())

In [ ]:
targets = ["IC50, mM", "CC50, mM", "SI"]
X = df.drop(columns=targets)

y = np.log1p(df["IC50, mM"])

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,random_state=42)

Опробуем линейную регрессию

In [ ]:
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score)
from sklearn.linear_model import LinearRegression

lr = LinearRegression() #нету значимых гиперпараметров
lr.fit(X_train, y_train)

pred = lr.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, pred))

mae = mean_absolute_error(y_test, pred)

r2 = r2_score(y_test, pred)

print("RMSE:", rmse)
print("MAE:", mae)
print("R2:", r2)

RMSE: 1.6046430723723093
MAE: 1.2691829801426218
R2: 0.3310576043471726


Линейная регрессия использована в качестве базовой модели. Полученное значение R2 = 0.33 показывает, что линейная модель объясняет около 33% изменчивости целевой переменной IC50.


Теперь посмотрим на модель случайного леса

In [ ]:
from sklearn.ensemble import RandomForestRegressor
rf = RandomForestRegressor(n_estimators=700) #оптимальное число деревьев для обучения модели, большее кол-во слабо влияет на метрики
rf.fit(X_train, y_train)

pred = rf.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, pred))

mae = mean_absolute_error(y_test, pred)

r2 = r2_score(y_test, pred)

print("RMSE:", rmse)
print("MAE:", mae)
print("R2:", r2)

RMSE: 1.4540206013275583
MAE: 1.1318245109473528
R2: 0.4507463521960443


Более высокое качество Random Forest по сравнению с линейной регрессией подтверждает наличие нелинейных зависимостей между молекулярными дескрипторами и показателем IC50.

Посмотрим на градиентный бустинг

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

gb = GradientBoostingRegressor(random_state=42, max_depth=10, min_samples_split=5) #глубина оптимальная, нету недообучения и риска переобучения
                                                                                    #min_samples_split делает деревья менее сложными борется с переобучением
gb.fit(X_train, y_train)

pred = gb.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, pred))
mae = mean_absolute_error(y_test, pred)
r2 = r2_score(y_test, pred)

print("Gradient Boosting")
print("RMSE:", rmse)
print("MAE:", mae)
print("R2:", r2)

Gradient Boosting
RMSE: 1.5873962065247733
MAE: 1.2130963697272357
R2: 0.34536004789468766


Итого среди 3 моделей наилучшие результаты показал случайный лес, что подтверждает нелинейный характер взаимосвязей дескрипторов с целевой переменной эффективности

Используем выделенные наиболее важные признаки дескрипторы который выделили в EDA и которые оказывают наибольшее влияние на показатели эффективности. Проверим насколько хорошо отдельно эти признаки описывают основную полезную информацию

In [ ]:
selected_features = [
    "VSA_EState8",
    "VSA_EState6",
    "VSA_EState4",
    "MolLogP",
    "qed",
    "FractionCSP3",
    "NumSaturatedHeterocycles",
    "NumSaturatedCarbocycles",
    "NumAliphaticCarbocycles",
]

X_small = df[selected_features]

X_train, X_test, y_train, y_test = train_test_split(
    X_small,
    y,
    test_size=0.2,
    random_state=42
)

Выбираем нашу лучшую модель

In [ ]:
rf = RandomForestRegressor(n_estimators=700)

rf.fit(X_train, y_train)

pred = rf.predict(X_test)

In [ ]:
rmse = np.sqrt(mean_squared_error(y_test, pred))
mae = mean_absolute_error(y_test, pred)
r2 = r2_score(y_test, pred)

print("Random Forest (selected features)")
print("RMSE:", rmse)
print("MAE:", mae)
print("R2:", r2)

Random Forest (selected features)
RMSE: 1.4967074767974913
MAE: 1.185270213441221
R2: 0.41802318083737455


Мы сократили число признаков с 211 всего до 9, при этом качество R2 упало лишь чуть более чем на 3%, что подтверждает высокую информативность наших 9 признаков, выявленных ранее на этапе EDA